# 05 - Production Benchmark and Calibration
Certificacion de modelos campeones con benchmarks ingenuos, calibracion de clasificacion y analisis de estabilidad.

In [1]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd
from scipy.stats import pearsonr

from sklearn.calibration import CalibratedClassifierCV
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier

base_dir = Path.cwd()
for root in [base_dir, *base_dir.parents]:
    if (root / "src").exists():
        if str(root) not in sys.path:
            sys.path.insert(0, str(root))
        break

from src.data_processing.build_dataset import get_training_features

warnings.filterwarnings("ignore")

dataset_path = None
for root in [base_dir, *base_dir.parents]:
    candidate = root / "data" / "processed" / "dataset_entrenamiento_final.csv"
    if candidate.exists():
        dataset_path = candidate
        break
if dataset_path is None:
    raise FileNotFoundError("dataset_entrenamiento_final.csv not found under data/processed")

df = pd.read_csv(dataset_path, parse_dates=["date"])
print("dataset:", dataset_path.resolve())
print("shape:", df.shape)

blacklist = [
    "precio_provincial_lag_1"
    "precio_provincial_lag_2"
    "precio_provincial_lag_3"
    "precio_vecinos_media_lag1"
    "precio_nacional_base_ma3"
    "precio_nacional_base_ma6"
    "precio_nacional_base_vol3"
    "precio_nacional_base_vol6"
]
target_candidates = ["precio_provincial_TARGET_H1", "precio_provincial_TARGET_H2", "precio_provincial_TARGET_H3"]
missing_targets = [t for t in target_candidates if t not in df.columns]
if missing_targets:
    raise ValueError(f"Missing target columns: {missing_targets}")
available_targets = target_candidates
horizons = [1, 2, 3]

split_date = pd.Timestamp("2021-01-01")
train_mask = df["date"] < split_date
test_mask = ~train_mask

train_df = df.loc[train_mask].copy()
test_df = df.loc[test_mask].copy()
print("train rows:", train_df.shape[0], "test rows:", test_df.shape[0])

identifiers = ["date", "provincia", "cereal_predominante"]
training_cols = get_training_features(df)
feature_cols = [
    c for c in training_cols
    if c in df.columns and c not in identifiers + available_targets
]
feature_cols = [c for c in feature_cols if c not in blacklist]

X_full = df[feature_cols].copy()
bool_cols = X_full.select_dtypes(include=["bool"]).columns
if len(bool_cols) > 0:
    X_full[bool_cols] = X_full[bool_cols].astype(int)

cat_cols = X_full.select_dtypes(include=["object", "category"]).columns.tolist()
if cat_cols:
    X_full = pd.get_dummies(X_full, columns=cat_cols, drop_first=False)

X_train = X_full.loc[train_mask].copy()
X_test = X_full.loc[test_mask].copy()
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

base_price_col = "precio_provincial_lag_1"
if base_price_col not in df.columns:
    raise ValueError("precio_provincial_lag_1 missing for targets")

def build_targets(horizon: int):
    target_reg = f"precio_provincial_TARGET_H{horizon}"
    y_train_reg = train_df[target_reg]
    y_test_reg = test_df[target_reg]
    base_train = train_df[base_price_col]
    base_test = test_df[base_price_col]
    y_train_clf = (y_train_reg - base_train > 0).astype(int)
    y_test_clf = (y_test_reg - base_test > 0).astype(int)
    return y_train_reg, y_test_reg, y_train_clf, y_test_clf

def regression_metrics(y_true, y_pred, base_series):
    aligned = pd.concat([y_true, base_series], axis=1).dropna()
    if aligned.empty:
        return {"Pearson": np.nan, "MAE": np.nan, "RMSE": np.nan, "DA": np.nan}
    y_true_clean = aligned.iloc[:, 0]
    base_clean = aligned.iloc[:, 1]
    y_pred_clean = pd.Series(y_pred, index=y_true.index).loc[aligned.index]
    mae = mean_absolute_error(y_true_clean, y_pred_clean)
    rmse = np.sqrt(mean_squared_error(y_true_clean, y_pred_clean))
    pearson = pearsonr(y_true_clean, y_pred_clean)[0] if y_true_clean.nunique() > 1 else np.nan
    da = (np.sign(y_pred_clean - base_clean) == np.sign(y_true_clean - base_clean)).mean()
    return {
        "Pearson": float(pearson) if pearson == pearson else np.nan,
        "MAE": float(mae),
        "RMSE": float(rmse),
        "DA": float(da),
    }

def classification_metrics(y_true, proba, pred):
    acc = accuracy_score(y_true, pred)
    da = acc
    auc = roc_auc_score(y_true, proba) if y_true.nunique() > 1 else np.nan
    return {
        "Accuracy": float(acc),
        "DA": float(da),
        "AUC": float(auc) if auc == auc else np.nan,
    }

def top_features_from_model(model, feature_names, top_k=5):
    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
        order = np.argsort(importances)[::-1][:top_k]
        return [feature_names[i] for i in order]
    if hasattr(model, "coef_"):
        coefs = np.ravel(model.coef_)
        order = np.argsort(np.abs(coefs))[::-1][:top_k]
        return [feature_names[i] for i in order]
    return []

dataset: C:\Users\marco\Desktop\Repos\DATAGIA-21\data\processed\dataset_entrenamiento_final.csv
shape: (7047, 78)
train rows: 5394 test rows: 1653


## 1. Stress test de senal (sin torta de girasol)
Entrenar un RF rapido excluyendo prepag2_torta de girasol_lag_1/2 y comparar Pearson contra el modelo base.

In [2]:
stress_exclude = {"prepag2_torta de girasol_lag_1", "prepag2_torta de girasol_lag_2"}

def train_fast_rf(X_train_in, y_train_in):
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)),
    ])
    model.fit(X_train_in, y_train_in)
    return model

h = 1
y_train_reg, y_test_reg, _, _ = build_targets(h)
base_test = test_df.loc[X_test.index, base_price_col]

train_mask_h = y_train_reg.notna()
test_mask_h = y_test_reg.notna()
X_train_h = X_train.loc[train_mask_h]
X_test_h = X_test.loc[test_mask_h]
y_train_h = y_train_reg.loc[train_mask_h]
y_test_h = y_test_reg.loc[test_mask_h]
base_test_h = base_test.loc[test_mask_h]

rf_full = train_fast_rf(X_train_h, y_train_h)
preds_full = rf_full.predict(X_test_h)
metrics_full = regression_metrics(y_test_h, preds_full, base_test_h)

kept_cols = [c for c in X_train_h.columns if c not in stress_exclude]
rf_stress = train_fast_rf(X_train_h[kept_cols], y_train_h)
preds_stress = rf_stress.predict(X_test_h[kept_cols])
metrics_stress = regression_metrics(y_test_h, preds_stress, base_test_h)

drop_ratio = 1 - (metrics_stress["Pearson"] / metrics_full["Pearson"]) if metrics_full["Pearson"] else np.nan

print("Pearson base (H1 RF):", metrics_full["Pearson"])
print("Pearson stress (sin torta):", metrics_stress["Pearson"])
print("Caida relativa Pearson:", drop_ratio)

if drop_ratio == drop_ratio and drop_ratio > 0.20:
    print("ALERTA: Caida > 20%. Torta de girasol parece proxy critico.")
else:
    print("OK: Pearson se mantiene dentro de rango aceptable.")

Pearson base (H1 RF): 0.7305953898128889
Pearson stress (sin torta): 0.7331924777299058
Caida relativa Pearson: -0.003554755413502031
OK: Pearson se mantiene dentro de rango aceptable.


## 2. Entrenamiento de campeones y calibracion
Regresion conserva campeones (RF/XGB). Clasificacion aplica calibracion sigmoide sobre el mejor modelo.

In [7]:
tscv = TimeSeriesSplit(n_splits=5)

def pearson_scorer(estimator, X, y):
    preds = estimator.predict(X)
    if y.nunique() <= 1:
        return 0.0
    return pearsonr(y, preds)[0]

reg_param_grids = {
    "RF": {
        "model__n_estimators": [300, 500, 700],
        "model__max_depth": [4, 6, 8, 12, None],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": ["sqrt", 0.6, 0.8],
    },
    "XGB": {
        "model__n_estimators": [200, 400, 600],
        "model__max_depth": [3, 5, 7],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__subsample": [0.6, 0.8, 1.0],
        "model__colsample_bytree": [0.6, 0.8, 1.0],
        "model__min_child_weight": [1, 5, 10],
    },
}

clf_param_grids = {
    "RF": {
        "model__n_estimators": [300, 500, 700],
        "model__max_depth": [4, 6, 8, 12, None],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": ["sqrt", 0.6, 0.8],
    },
    "XGB": {
        "model__n_estimators": [200, 400, 600],
        "model__max_depth": [3, 5, 7],
        "model__learning_rate": [0.01, 0.05, 0.1],
        "model__subsample": [0.6, 0.8, 1.0],
        "model__colsample_bytree": [0.6, 0.8, 1.0],
        "model__min_child_weight": [1, 5, 10],
    },
}

reg_champions = {}
clf_champions = {}
clf_calibrated = {}

for h in horizons:
    y_train_reg, y_test_reg, y_train_clf, y_test_clf = build_targets(h)
    base_train = train_df.loc[X_train.index, base_price_col]
    base_test = test_df.loc[X_test.index, base_price_col]

    train_mask_h = y_train_reg.notna() & base_train.notna()
    test_mask_h = y_test_reg.notna() & base_test.notna()
    X_train_h = X_train.loc[train_mask_h]
    X_test_h = X_test.loc[test_mask_h]
    y_train_h = y_train_reg.loc[train_mask_h]
    y_test_h = y_test_reg.loc[test_mask_h]
    base_test_h = base_test.loc[test_mask_h]

    reg_models = {
        "RF": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(random_state=42, n_jobs=-1)),
        ]),
        "XGB": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBRegressor(random_state=42, n_jobs=-1, objective="reg:squarederror")),
        ]),
    }

    best_by_name = {}
    for name, model in reg_models.items():
        search = RandomizedSearchCV(
            model,
            param_distributions=reg_param_grids[name],
            n_iter=20,
            scoring=pearson_scorer,
            cv=tscv,
            random_state=42,
            n_jobs=-1,
        )
        search.fit(X_train_h, y_train_h)
        best_by_name[name] = search.best_estimator_

    best_name = None
    best_metrics = None
    best_model = None
    for name, model in best_by_name.items():
        preds = model.predict(X_test_h)
        metrics = regression_metrics(y_test_h, preds, base_test_h)
        if best_metrics is None or metrics["Pearson"] > best_metrics["Pearson"]:
            best_name = name
            best_metrics = metrics
            best_model = model

    reg_champions[h] = {
        "model": best_name,
        "metrics": best_metrics,
        "estimator": best_model,
    }

    train_mask_clf = y_train_clf.notna()
    test_mask_clf = y_test_clf.notna()
    X_train_c = X_train.loc[train_mask_clf]
    X_test_c = X_test.loc[test_mask_clf]
    y_train_c = y_train_clf.loc[train_mask_clf]
    y_test_c = y_test_clf.loc[test_mask_clf]

    clf_models = {
        "RF": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(random_state=42, n_jobs=-1)),
        ]),
        "XGB": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(random_state=42, n_jobs=-1, eval_metric="logloss")),
        ]),
    }

    best_by_name = {}
    for name, model in clf_models.items():
        search = RandomizedSearchCV(
            model,
            param_distributions=clf_param_grids[name],
            n_iter=20,
            scoring="roc_auc",
            cv=tscv,
            random_state=42,
            n_jobs=-1,
        )
        search.fit(X_train_c, y_train_c)
        best_by_name[name] = search.best_estimator_

    best_name = None
    best_metrics = None
    best_model = None
    for name, model in best_by_name.items():
        proba = model.predict_proba(X_test_c)[:, 1]
        pred = (proba >= 0.5).astype(int)
        metrics = classification_metrics(y_test_c, proba, pred)
        if best_metrics is None or metrics["DA"] > best_metrics["DA"]:
            best_name = name
            best_metrics = metrics
            best_model = model

    clf_champions[h] = {
        "model": best_name,
        "metrics": best_metrics,
        "estimator": best_model,
    }

    calib = CalibratedClassifierCV(estimator=best_model, method="sigmoid", cv=TimeSeriesSplit(n_splits=3))
    calib.fit(X_train_c, y_train_c)
    proba_cal = calib.predict_proba(X_test_c)[:, 1]
    pred_cal = (proba_cal >= 0.5).astype(int)
    metrics_cal = classification_metrics(y_test_c, proba_cal, pred_cal)
    clf_calibrated[h] = {
        "model": best_name,
        "metrics": metrics_cal,
        "estimator": calib,
    }

print("Reg champions:", {h: reg_champions[h]["model"] for h in horizons})
print("Clf champions:", {h: clf_champions[h]["model"] for h in horizons})
print("Clf calibrated AUC:", {h: clf_calibrated[h]["metrics"]["AUC"] for h in horizons})

Reg champions: {1: 'RF', 2: 'RF', 3: 'RF'}
Clf champions: {1: 'RF', 2: 'RF', 3: 'RF'}
Clf calibrated AUC: {1: 0.8576100562401934, 2: 0.8200274725274725, 3: 0.8702313832840836}


## 3. Benchmarking contra estrategias ingenuas
Comparar Directional Accuracy en clasificacion y MAE/RMSE en regresion contra persistencia y reglas simples.

In [8]:
bench_rows_clf = []
bench_rows_reg = []

for h in horizons:
    y_train_reg, y_test_reg, y_train_clf, y_test_clf = build_targets(h)
    base_test = test_df.loc[X_test.index, base_price_col]
    test_mask_h = y_test_reg.notna() & base_test.notna()
    y_test_h = y_test_reg.loc[test_mask_h]
    base_test_h = base_test.loc[test_mask_h]

    # Reg benchmark
    naive_pred = base_test_h
    naive_metrics = regression_metrics(y_test_h, naive_pred, base_test_h)
    model_pred = reg_champions[h]["estimator"].predict(X_test.loc[test_mask_h])
    model_metrics = regression_metrics(y_test_h, model_pred, base_test_h)
    bench_rows_reg.append({
        "horizon": h,
        "model": reg_champions[h]["model"],
        "MAE_model": model_metrics["MAE"],
        "RMSE_model": model_metrics["RMSE"],
        "DA_model": model_metrics["DA"],
        "MAE_naive": naive_metrics["MAE"],
        "RMSE_naive": naive_metrics["RMSE"],
        "DA_naive": naive_metrics["DA"],
    })

    # Clf benchmark
    test_mask_c = y_test_clf.notna()
    y_test_c = y_test_clf.loc[test_mask_c]
    X_test_c = X_test.loc[test_mask_c]

    proba_cal = clf_calibrated[h]["estimator"].predict_proba(X_test_c)[:, 1]
    pred_cal = (proba_cal >= 0.5).astype(int)
    metrics_cal = classification_metrics(y_test_c, proba_cal, pred_cal)

    pred_persist = np.zeros_like(y_test_c.values)
    pred_up = np.ones_like(y_test_c.values)
    pred_down = np.zeros_like(y_test_c.values)

    da_persist = accuracy_score(y_test_c, pred_persist)
    da_up = accuracy_score(y_test_c, pred_up)
    da_down = accuracy_score(y_test_c, pred_down)

    bench_rows_clf.append({
        "horizon": h,
        "model": clf_calibrated[h]["model"],
        "DA_model": metrics_cal["DA"],
        "AUC_model": metrics_cal["AUC"],
        "DA_persist": da_persist,
        "DA_solo_sube": da_up,
        "DA_solo_baja": da_down,
    })

bench_reg_df = pd.DataFrame(bench_rows_reg)
bench_clf_df = pd.DataFrame(bench_rows_clf)
bench_reg_df
bench_clf_df

,horizon,model,DA_model,AUC_model,DA_persist,DA_solo_sube,DA_solo_baja
0,1,RF,0.714459,0.857610,0.529946,0.470054,0.529946
1,2,RF,0.738052,0.820027,0.559589,0.440411,0.559589
2,3,RF,0.730188,0.870231,0.588022,0.411978,0.588022


## 4. Estabilidad por cereal
Calcular Pearson y DA por cereal_predominante para cada horizonte.

In [9]:
cereals = [c for c in df["cereal_predominante"].dropna().unique() if c in ["Trigo", "Cebada", "Maiz", "Maiz"]]
if not cereals:
    cereals = df["cereal_predominante"].dropna().unique().tolist()

stability_rows_reg = []
stability_rows_clf = []

for h in horizons:
    y_train_reg, y_test_reg, y_train_clf, y_test_clf = build_targets(h)
    base_test = test_df.loc[X_test.index, base_price_col]
    test_mask_h = y_test_reg.notna() & base_test.notna()
    y_test_h = y_test_reg.loc[test_mask_h]
    base_test_h = base_test.loc[test_mask_h]
    X_test_h = X_test.loc[test_mask_h]

    reg_model = reg_champions[h]["estimator"]
    reg_pred = pd.Series(reg_model.predict(X_test_h), index=y_test_h.index)

    test_c_mask = y_test_clf.notna()
    y_test_c = y_test_clf.loc[test_c_mask]
    X_test_c = X_test.loc[test_c_mask]
    clf_model = clf_calibrated[h]["estimator"]
    proba_c = pd.Series(clf_model.predict_proba(X_test_c)[:, 1], index=y_test_c.index)
    pred_c = (proba_c >= 0.5).astype(int)

    for cereal in cereals:
        idx_reg = test_df.loc[test_mask_h].index[test_df.loc[test_mask_h, "cereal_predominante"] == cereal]
        if len(idx_reg) > 0:
            y_true_reg = y_test_h.loc[idx_reg]
            y_pred_reg = reg_pred.loc[idx_reg]
            base_reg = base_test_h.loc[idx_reg]
            reg_metrics = regression_metrics(y_true_reg, y_pred_reg, base_reg)
            stability_rows_reg.append({
                "horizon": h,
                "cereal": cereal,
                "Pearson": reg_metrics["Pearson"],
                "DA": reg_metrics["DA"],
            })

        idx_clf = test_df.loc[test_c_mask].index[test_df.loc[test_c_mask, "cereal_predominante"] == cereal]
        if len(idx_clf) > 0:
            y_true_clf = y_test_c.loc[idx_clf]
            proba_clf = proba_c.loc[idx_clf]
            pred_clf = pred_c.loc[idx_clf]
            metrics_clf = classification_metrics(y_true_clf, proba_clf, pred_clf)
            pearson_clf = pearsonr(y_true_clf, proba_clf)[0] if y_true_clf.nunique() > 1 else np.nan
            stability_rows_clf.append({
                "horizon": h,
                "cereal": cereal,
                "Pearson": pearson_clf if pearson_clf == pearson_clf else np.nan,
                "DA": metrics_clf["DA"],
                "AUC": metrics_clf["AUC"],
            })

stability_reg_df = pd.DataFrame(stability_rows_reg)
stability_clf_df = pd.DataFrame(stability_rows_clf)
stability_reg_df
stability_clf_df

,horizon,cereal,Pearson,DA,AUC
0,1,cebada,0.624291,0.732714,0.869745
1,1,trigo,0.606718,0.688596,0.853856
2,2,cebada,0.543602,0.692466,0.815363
3,2,trigo,0.591298,0.802632,0.831747
4,3,cebada,0.627627,0.711042,0.873780
5,3,trigo,0.629228,0.757310,0.867992


## 5. Reporte de certificacion
Guardar CERTIFICACION_DATAGIA_PROD.md en reports/.

In [10]:
report_root = None
for root in [base_dir, *base_dir.parents]:
    candidate = root / "reports"
    if candidate.exists():
        report_root = candidate
        break
if report_root is None:
    report_root = base_dir / "reports"
    report_root.mkdir(parents=True, exist_ok=True)

report_path = report_root / "CERTIFICACION_DATAGIA_PROD.md"

reg_beats = []
for row in bench_reg_df.to_dict(orient="records"):
    beats = (row["MAE_model"] < row["MAE_naive"]) and (row["RMSE_model"] < row["RMSE_naive"]) and (row["DA_model"] > row["DA_naive"])
    reg_beats.append({"horizon": row["horizon"], "model": row["model"], "beats_naive": beats})

clf_beats = []
for row in bench_clf_df.to_dict(orient="records"):
    max_baseline = max(row["DA_persist"], row["DA_solo_sube"], row["DA_solo_baja"])
    beats = row["DA_model"] > max_baseline
    clf_beats.append({"horizon": row["horizon"], "model": row["model"], "beats_naive": beats})

lines = [
    "# CERTIFICACION_DATAGIA_PROD",
    "",
    "## Resumen",
    "Certificacion de 6 modelos campeones con calibracion y benchmarks ingenuos.",
    "",
    "## Stress test (sin torta de girasol)",
    f"Pearson base (H1 RF): {metrics_full['Pearson']}",
    f"Pearson stress: {metrics_stress['Pearson']}",
    f"Caida relativa: {drop_ratio}",
    "",
    "## Benchmark regresion (vs retorno cero)",
    bench_reg_df.to_markdown(index=False),
    "",
    "## Benchmark clasificacion (vs baselines)",
    bench_clf_df.to_markdown(index=False),
    "",
    "## Estabilidad por cereal (regresion)",
    stability_reg_df.to_markdown(index=False),
    "",
    "## Estabilidad por cereal (clasificacion)",
    stability_clf_df.to_markdown(index=False),
    "",
    "## Modelos que baten baselines",
    "Regresion:",
    pd.DataFrame(reg_beats).to_markdown(index=False),
    "",
    "Clasificacion:",
    pd.DataFrame(clf_beats).to_markdown(index=False),
    "",
]

report_path.write_text("\n".join(lines), encoding="utf-8")
print("Reporte guardado en:", report_path.resolve())

Reporte guardado en: C:\Users\marco\Desktop\Repos\DATAGIA-21\notebooks\training\reports\CERTIFICACION_DATAGIA_PROD.md
